# Polarisation synthesis parity on San Francisco ALOS-1 data

This notebook runs the legacy C-PolSARpro `polar_synt` executable and the Python implementation from the same San Francisco ALOS-1 scattering matrix. It checks both Pauli and Sinclair output conventions at nonzero orientation and ellipticity angles, including each implementation's S-to-T3 conversion. The C program writes raw float rasters despite calling its optional single-channel output a BMP.

In [ ]:
%load_ext autoreload
%autoreload 2

import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dask.diagnostics import ProgressBar

from polsarpro.dev.io import read_psp_bin
from polsarpro.dev.metrics import summarize_metrics, visualize_errors
from polsarpro.io import open_netcdf_beam
from polsarpro.polarisation import polarisation_synthesis

c_executable = Path("/home/c_psp/Soft/bin/data_process_sngl/polar_synt.exe")
input_file = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")
input_dir = Path("/data/psp/SAN_FRANCISCO_ALOS1")
output_dir = Path("/data/psp/res/polarisation_synthesis_alos1_c")
output_dir.mkdir(parents=True, exist_ok=True)

phi = 17.0
tau = -11.0

## Load the common scattering-matrix input

Python opens the ALOS-1 scattering matrix from NetCDF while C reads its equivalent PolSARpro S2 binary bands. The supplied valid-pixel mask is applied to Python and passed to C; C's zero-valued invalid output pixels are converted to NaN before comparison to match the Python dataset convention.

In [ ]:
S = open_netcdf_beam(input_file)
nrows, ncols = S.sizes["y"], S.sizes["x"]
mask_file = input_dir / "mask_valid_pixels.bin"
valid = read_psp_bin(mask_file).astype(bool)
# S = S.where(valid)

## Run C-PolSARpro for both channel conventions

In [ ]:
files_by_basis = {}

for basis in ("pauli", "sinclair"):
    print(f"Basis: {basis}")
    files = {
        channel: output_dir / f"{basis}_{channel}.bin"
        for channel in ("red", "green", "blue")
    }
    command = [
        str(c_executable),
        "-id", str(input_dir),
        "-iodf", "S2",
        "-ofr", "0",
        "-ofc", "0",
        "-fnr", str(nrows),
        "-fnc", str(ncols),
        "-phi", str(phi),
        "-tau", str(tau),
        "-rgb", "1",
        "-rgbf", basis,
        "-bf", str(files["blue"]),
        "-rf", str(files["red"]),
        "-gf", str(files["green"]),
        "-bmp", "0",
        "-mask", str(mask_file),
        "-errf", str(output_dir / "memory_error.txt"),
    ]
    subprocess.run(command)
    files_by_basis[basis] = files

## Run Python for both channel conventions

In [ ]:
out_py_by_basis = {}
with ProgressBar():
    for basis in ("pauli", "sinclair"):
        print(f"Basis: {basis}")
        out_py_by_basis[basis] = polarisation_synthesis(
            S, phi=phi, tau=tau, basis=basis
        ).compute()

## Numerical comparison

The Python implementation evaluates the same equations lazily and stores the final channels as float32. Small finite differences are expected because the C routine stores several intermediate values as float32 while its trigonometric functions evaluate in double precision.

In [ ]:
out_c_by_basis = {
    basis: {
        channel: np.where(
            valid,
            np.fromfile(path, dtype=np.float32).reshape(nrows, ncols),
            np.nan,
        )
        for channel, path in files.items()
    }
    for basis, files in files_by_basis.items()
}

metrics = {
    basis: summarize_metrics(
        out_py_by_basis[basis].to_dataset(dim="band"),
        out_c_by_basis[basis],
        verbose=False,
    )
    for basis in out_py_by_basis
}
pd.concat(metrics, names=["basis", "channel"] )

In [ ]:
for basis, out_py in out_py_by_basis.items():
    out_c = out_c_by_basis[basis]
    print(f"{basis}: maximum absolute differences")
    for channel in out_py.band.values:
        difference = np.nanmax(
            np.abs(out_py.sel(band=channel).values - out_c[channel])
        )
        print(f"  {channel}: {difference:.8g}")

## Inspect spatial error patterns

In [ ]:
basis = "sinclair"
out_py = out_py_by_basis[basis]
out_c = out_c_by_basis[basis]
visualize_errors(
    out_py.to_dataset(dim="band"), out_c, sub_az=8, sub_rg=1
)